# Test notebook

The purpose of this notebook is to test an equation and compare them with the baselines: Burton, MBR, and DDM1, 2 and 3. We will also plot each storm and get the metrics for the equation.

The only cell that we have to modify is the following one, where we can change the features, the mode (template or default) and the output directory for the plots.
Raw EQ is the equation that we want to test, the raw version generated from the train_script.py file.

In [1]:
import os

FEATURES1 = ["Vp", "Np", "Bzsouth", "Bmag", "DST"]
RAW_EQ_1 = 'g = (#3 - -2.152752) * ((((4.7333016 - #2) * #3) - #1) / 501.94562); d = square(1.4046433 - (#1 * 0.015329162))'
FEATURES2 = ["P_dyn", "VBs", "epsilon", "DST"]
RAW_EQ_2 = "g = (#2 * -0.0010713526) * sqrt(#1 + 1.32442); d = square((#1 * 0.01838707) - 0.5912093)"
MODE = "template"  # 'template' or 'default'
OUTPUT_DIR = "template_primitive_features_fig4"

start_eq_number = 13

output_folder = OUTPUT_DIR
# Count number of existing subfolders
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

import sympy as sp
from tqdm import tqdm

from sympy.printing import latex

# Internal module imports
import storm_dates
import baseline_models

# from evaluation_engine import UnifiedModel, simulate_storm, compute_features
from evaluation_engine import EquationModel, simulate_storm
from train_script import load_and_preprocess, compute_features

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/spacepy/time.py:2448: UserWarning: Leapseconds may be out of date. Use spacepy.toolbox.update(leapsecs=True)
  _read_leaps()


In [3]:
raw_data = load_and_preprocess()
data = compute_features(raw_data)

In [4]:
def predict_and_plot_storm(
    model1, eq1, model2, eq2, start, end, storm_df, storm_id, save_path, eq1_ind=13, eq2_ind=21
):
    # 1. Generate Predictions
    y_true = storm_df[start:end]["DST"].values
    colors = ["blue", "yellow", "green", "orange", "purple", "cyan", "magenta"]
    res_eqs = []

    metrics_info = []
    
    res_eq1 = simulate_storm(model1, storm_df)
    res_eq1 = res_eq1[start:end]["DST_pred"].values
    m_eq1 = baseline_models.get_all_metrics_dict(y_true, res_eq1)
    metrics_info.append(m_eq1)
    res_eqs.append(res_eq1)

    res_eq2 = simulate_storm(model2, storm_df)
    res_eq2 = res_eq2[start:end]["DST_pred"].values
    m_eq2 = baseline_models.get_all_metrics_dict(y_true, res_eq2)
    metrics_info.append(m_eq2)

    res_eqs.append(res_eq2)

    fig, axs = plt.subplots(1, 3, figsize=(24, 9), constrained_layout=True)

    # Column 1: Time Series
    axs[0].plot(
        storm_df[start:end].index,
        y_true,
        color="black",
        label="Observed",
        alpha=0.6,
        linewidth=2,
    )
    
    axs[0].plot(
        storm_df[start:end].index,
        res_eqs[0],
        color=colors[0],
        linestyle="--",
        # label=f"Equation {eq_index + eq_start_index}",
        label=None,
        linewidth=1.5,
    )
    
    axs[0].plot(
            storm_df[start:end].index,
            res_eqs[1],
            color=colors[1],
            linestyle="--",
            # label=f"Equation {eq_index + eq_start_index}",
            label=None,
            linewidth=1.5,
        )

    axs[0].tick_params(axis="both", which="major", labelsize=20)
    axs[0].tick_params(axis="both", which="minor", labelsize=18)
    axs[0].legend(fontsize=20)
    axs[0].set_ylabel("Dst (nT)", fontsize=20)
    axs[0].set_xlabel("Date", fontsize=20)
    axs[0].grid(True)
    axs[0].set_xlim(start, end)
    axs[0].set_title("Storm Reconstruction", fontsize=24)

    axs[0].xaxis.set_major_locator(MultipleLocator(2))
    
    if len(axs[0].xaxis.get_ticklabels()) > 7:    
        for label in axs[0].xaxis.get_ticklabels()[1::2]:
            label.set_visible(False)

    diffs = []

    for eq_index, res_eq in enumerate(res_eqs):
        diff_eq = res_eq - y_true
        diffs.append(diff_eq)

        # axs[1].plot(storm_df[start:end].index, diff_eq, color=colors[eq_index], label=f"Equation {eq_index + eq_start_index} Error")
        axs[1].plot(
            storm_df[start:end].index, diff_eq, color=colors[eq_index], label=None
        )

    axs[1].axhline(0, color="black", linestyle="--")

    title_metrics = f"Error Comparison\n"

    for eq_index, m_eq in enumerate(metrics_info):
        title_metrics += f"Eq {[eq1_ind, eq2_ind][eq_index]} ({colors[eq_index]}): MAE={m_eq['MAE']:.2f}, RMSE={m_eq['RMSE']:.2f}, R²={m_eq['R2']:.3f}, BFE={m_eq['BFE']:.3f}\n"

    # axs[1].set_title(title_metrics, fontsize=18)
    axs[1].set_title("Equation error", fontsize=24)
    axs[1].set_ylabel("Error (nT)", fontsize=20)
    axs[1].set_xlabel("Date", fontsize=20)
    axs[1].grid(True)
    axs[1].set_xlim(start, end)
    axs[1].tick_params(axis="both", which="major", labelsize=20)
    axs[1].tick_params(axis="both", which="minor", labelsize=18)

    axs[1].xaxis.set_major_locator(MultipleLocator(2))
    
    if len(axs[1].xaxis.get_ticklabels()) > 7:
        for label in axs[1].xaxis.get_ticklabels()[1::2]:
            label.set_visible(False)

    # Column 3: BFE
    baseline_models.plot_evaluation_bfe_multi(
        axs[2],
        y_true,
        res_eqs,
        [f"Equation {eq1_ind}", f"Equation {eq2_ind}"],
        [colors[0], colors[1]],
        fontsize=20,
        plot_legend=False,
    )

    string_title = f"Storm {storm_id} Reconstruction\n{title_metrics}"
    fig.suptitle(string_title, fontsize=18)
    plt.savefig(save_path)
    plt.close()

In [5]:
def save_prediction_data(model, eqs, start, end, storm_df, output_path):
    """
    Generates and saves a CSV with observed and predicted DST and dDST/dt.
    """
    # 1. Observed Data
    # Real dDST is calculated as the difference to the next hour
    real_dst = storm_df[start:end]["DST"].values
    real_ddst = storm_df[start:end]["DST"].diff().shift(-1).values

    # 2. Equation Predictions
    # We need the iterative predictions for DST
    pred_dst_eqs = []
    for eq_index, eq in enumerate(eqs):
        pred_dst_eq = simulate_storm(model[eq], storm_df)
        if model[eq].is_template:
            pred_dst_eq = pred_dst_eq[start:end][
                ["DST_pred", "dDST", "injection_component", "decay_component"]
            ]
        else:
            pred_dst_eq = pred_dst_eq[start:end][["DST_pred", "dDST"]]
        pred_dst_eqs.append(pred_dst_eq)
        
    # 3. Baseline Predictions (Burton & OBM)
    
    # 4. Construct Comprehensive DataFrame

    if model[eq].is_template:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Injection_Component": pred_dst_eq["injection_component"].values,
                "Decay_Component": pred_dst_eq["decay_component"].values,
                
            }
        ).set_index("Timestamp")
        
        for eq_index, pred_dst_eq in enumerate(pred_dst_eqs):
            results_df[f"Pred_DST_Equation_{eq_index+1}"] = pred_dst_eq["DST_pred"].values
            results_df[f"Pred_dDST_dt_Equation_{eq_index+1}"] = pred_dst_eq["dDST"].values
            results_df[f"Injection_Component_{eq_index+1}"] = pred_dst_eq["injection_component"].values
            results_df[f"Decay_Component_{eq_index+1}"] = pred_dst_eq["decay_component"].values
        
    else:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,                
            }
        ).set_index("Timestamp")
        
        for eq_index, pred_dst_eq in enumerate(pred_dst_eqs):
            results_df[f"Pred_DST_Equation_{eq_index+1}"] = pred_dst_eq["DST_pred"].values
            results_df[f"Pred_dDST_dt_Equation_{eq_index+1}"] = pred_dst_eq["dDST"].values


    results_df.to_csv(output_path)
    return results_df

## Test storms

In [6]:
storms = []
storm_indices = []
models = {}

model1 = EquationModel(RAW_EQ_1, FEATURES1, is_template=MODE == "template")
models[RAW_EQ_1] = model1

model2 = EquationModel(RAW_EQ_2, FEATURES2, is_template=MODE == "template")
models[RAW_EQ_2] = model2

test_storms = storm_dates.TEST_STORMS_SYMBOLIC_REGRESSION

for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        model1,
        RAW_EQ_1,
        model2,
        RAW_EQ_2,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    '''
    storms.append(
        save_prediction_data(
            models, RAW_EQS, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)
    '''
    
for eq_index, RAW_EQ in enumerate([RAW_EQ_1, RAW_EQ_2]):
    with open(os.path.join(OUTPUT_DIR, f'equation_{eq_index+1}.txt'), 'w') as f:
        f.write(f'Equation: {RAW_EQ}\n')            
        f.write(f'LaTeX: {latex(models[RAW_EQ].latex_str())}\n')


  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:08<00:00,  2.48it/s]
